# Week 14: Deep Learning Training Tricks
# 第 14 週：深度學習訓練技巧（學習率排程、早停、資料增強）

## 學習目標 Learning Objectives
1. 從零實作 SGD、Momentum、RMSProp、Adam 優化器並比較路徑
2. 觀察不同學習率對模型訓練曲線的影響
3. 實作與視覺化五種學習率排程策略
4. 模擬早停機制，比較有/無早停的效果
5. 視覺化不同權重初始化方法的分布差異
6. 使用 numpy/scipy 實作影像資料增強
7. 模擬不同 Batch Size 對收斂的影響
8. 視覺化梯度爆炸與梯度裁剪
9. 綜合比較所有訓練技巧的效果

**環境需求 Required packages**：numpy, matplotlib, sklearn, scipy

In [ ]:
# 匯入必要套件 Import required packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import make_moons, make_classification, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from scipy.ndimage import rotate, shift, gaussian_filter

# 設定中文字型
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

np.random.seed(42)

print('所有套件匯入成功！ All packages imported successfully!')

---
## Part 1: 優化器比較 Optimizer Comparison

我們在 Rosenbrock 函數上比較四種優化器的路徑：

$$f(x, y) = (a - x)^2 + b(y - x^2)^2, \quad a=1, b=100$$

- **SGD**: $\theta \leftarrow \theta - \eta \nabla L$
- **Momentum**: $v \leftarrow \beta v + \nabla L$, $\theta \leftarrow \theta - \eta v$
- **RMSProp**: $s \leftarrow \gamma s + (1-\gamma)(\nabla L)^2$, $\theta \leftarrow \theta - \frac{\eta}{\sqrt{s+\epsilon}} \nabla L$
- **Adam**: combines Momentum + RMSProp with bias correction

In [ ]:
# Rosenbrock 函數及其梯度
def rosenbrock(x, y, a=1, b=100):
    return (a - x)**2 + b * (y - x**2)**2

def rosenbrock_grad(x, y, a=1, b=100):
    dx = -2 * (a - x) + b * 2 * (y - x**2) * (-2 * x)
    dy = b * 2 * (y - x**2)
    return np.array([dx, dy])

# 四種優化器實作
def optimize_sgd(grad_fn, x0, lr=0.0005, steps=500):
    path = [x0.copy()]
    x = x0.copy()
    for _ in range(steps):
        g = grad_fn(x[0], x[1])
        x = x - lr * g
        path.append(x.copy())
    return np.array(path)

def optimize_momentum(grad_fn, x0, lr=0.0005, beta=0.9, steps=500):
    path = [x0.copy()]
    x = x0.copy()
    v = np.zeros_like(x)
    for _ in range(steps):
        g = grad_fn(x[0], x[1])
        v = beta * v + g
        x = x - lr * v
        path.append(x.copy())
    return np.array(path)

def optimize_rmsprop(grad_fn, x0, lr=0.005, gamma=0.9, eps=1e-8, steps=500):
    path = [x0.copy()]
    x = x0.copy()
    s = np.zeros_like(x)
    for _ in range(steps):
        g = grad_fn(x[0], x[1])
        s = gamma * s + (1 - gamma) * g**2
        x = x - lr * g / (np.sqrt(s) + eps)
        path.append(x.copy())
    return np.array(path)

def optimize_adam(grad_fn, x0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, steps=500):
    path = [x0.copy()]
    x = x0.copy()
    m = np.zeros_like(x)
    v = np.zeros_like(x)
    for t in range(1, steps + 1):
        g = grad_fn(x[0], x[1])
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g**2
        m_hat = m / (1 - beta1**t)
        v_hat = v / (1 - beta2**t)
        x = x - lr * m_hat / (np.sqrt(v_hat) + eps)
        path.append(x.copy())
    return np.array(path)

# 起始點
x0 = np.array([-1.5, 2.0])
n_steps = 800

# 執行各優化器
path_sgd = optimize_sgd(rosenbrock_grad, x0, lr=0.0003, steps=n_steps)
path_mom = optimize_momentum(rosenbrock_grad, x0, lr=0.0003, steps=n_steps)
path_rms = optimize_rmsprop(rosenbrock_grad, x0, lr=0.003, steps=n_steps)
path_adam = optimize_adam(rosenbrock_grad, x0, lr=0.01, steps=n_steps)

# 繪製等高線圖和優化路徑
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

xx, yy = np.meshgrid(np.linspace(-2, 2, 300), np.linspace(-1, 3, 300))
zz = rosenbrock(xx, yy)

paths = [
    (path_sgd, 'SGD', '#E74C3C'),
    (path_mom, 'SGD + Momentum', '#3498DB'),
    (path_rms, 'RMSProp', '#2ECC71'),
    (path_adam, 'Adam', '#9B59B6')
]

for ax, (path, name, color) in zip(axes.flat, paths):
    ax.contour(xx, yy, zz, levels=np.logspace(-1, 3.5, 30), cmap='coolwarm', alpha=0.6)
    ax.plot(path[:, 0], path[:, 1], '-', color=color, linewidth=1.5, alpha=0.8)
    ax.plot(path[0, 0], path[0, 1], 'ko', markersize=8, label='Start')
    ax.plot(1, 1, 'r*', markersize=15, label='Minimum (1,1)')
    ax.plot(path[-1, 0], path[-1, 1], 's', color=color, markersize=8, label='End')
    final_loss = rosenbrock(path[-1, 0], path[-1, 1])
    ax.set_title(f'{name}\nFinal loss: {final_loss:.4f} ({len(path)} steps)', fontsize=12)
    ax.set_xlabel('x', fontsize=10)
    ax.set_ylabel('y', fontsize=10)
    ax.legend(fontsize=8, loc='upper left')
    ax.set_xlim(-2, 2)
    ax.set_ylim(-1, 3)

plt.suptitle('Optimizer Comparison on Rosenbrock Function\n'
             '優化器在 Rosenbrock 函數上的比較', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('觀察重點 Key Observations:')
print('  SGD:      最直接但沿狹長山谷前進緩慢')
print('  Momentum: 加速沿山谷方向的移動')
print('  RMSProp:  自適應學習率，不同方向不同步幅')
print('  Adam:     結合 Momentum + RMSProp，通常收斂最快')

---
## Part 2: 學習率效果 Learning Rate Effect

學習率 $\eta$ 是最重要的超參數之一：
$$\theta_{t+1} = \theta_t - \eta \cdot \nabla_\theta L$$

- **太大** → 訓練不穩定，損失震盪甚至發散
- **太小** → 收斂速度極慢，可能困在局部最小值
- **剛好** → 快速且穩定地收斂

In [ ]:
# 使用 sklearn MLPClassifier 比較不同學習率
np.random.seed(42)
X_lr, y_lr = make_moons(n_samples=500, noise=0.25, random_state=42)
X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(
    X_lr, y_lr, test_size=0.3, random_state=42
)
scaler_lr = StandardScaler()
X_train_lr = scaler_lr.fit_transform(X_train_lr)
X_test_lr = scaler_lr.transform(X_test_lr)

learning_rates = [0.0001, 0.001, 0.01, 0.1, 1.0]
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(learning_rates)))

results_lr = []
for lr_val, color in zip(learning_rates, colors):
    mlp = MLPClassifier(
        hidden_layer_sizes=(32, 16), max_iter=300,
        learning_rate_init=lr_val, learning_rate='constant',
        solver='sgd', random_state=42, batch_size=32
    )
    mlp.fit(X_train_lr, y_train_lr)
    
    train_acc = mlp.score(X_train_lr, y_train_lr)
    test_acc = mlp.score(X_test_lr, y_test_lr)
    results_lr.append((lr_val, train_acc, test_acc))
    
    # 訓練曲線 Training curve
    axes[0].plot(mlp.loss_curve_, color=color, linewidth=1.5,
                 label=f'LR={lr_val}')

axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Training Loss', fontsize=11)
axes[0].set_title('Training Loss Curves with Different Learning Rates\n'
                   '不同學習率的訓練曲線', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 1.5)

# 右圖：準確率比較
lr_labels = [f'{lr}' for lr in learning_rates]
train_accs = [r[1] for r in results_lr]
test_accs = [r[2] for r in results_lr]

x_pos = np.arange(len(learning_rates))
width = 0.35
axes[1].bar(x_pos - width/2, train_accs, width, label='Train Acc', color='steelblue', alpha=0.8)
axes[1].bar(x_pos + width/2, test_accs, width, label='Test Acc', color='coral', alpha=0.8)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(lr_labels)
axes[1].set_xlabel('Learning Rate', fontsize=11)
axes[1].set_ylabel('Accuracy', fontsize=11)
axes[1].set_title('Train/Test Accuracy vs Learning Rate\n學習率 vs 準確率', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_ylim(0.5, 1.05)

plt.tight_layout()
plt.show()

for lr, train_a, test_a in results_lr:
    print(f'LR={lr:<8} Train Acc={train_a:.4f}  Test Acc={test_a:.4f}')

---
## Part 3: 學習率排程 Learning Rate Schedules

在訓練中**動態調整學習率**，初期大步探索，後期小步精調。

常見策略：
- **Constant** (基線)
- **Step Decay**: $\eta_t = \eta_0 \times \gamma^{\lfloor t/S \rfloor}$
- **Exponential Decay**: $\eta_t = \eta_0 \times \gamma^t$
- **Cosine Annealing**: $\eta_t = \eta_{min} + \frac{1}{2}(\eta_{max} - \eta_{min})(1 + \cos(\frac{t}{T}\pi))$
- **Warmup + Cosine**: 先線性增加再餘弦衰減

In [ ]:
# 實作並視覺化五種學習率排程
total_epochs = 100
base_lr = 0.1
min_lr = 1e-5
epochs = np.arange(total_epochs)

# 1. Constant
lr_constant = np.ones(total_epochs) * base_lr

# 2. Step Decay (every 30 epochs, multiply by 0.1)
lr_step = np.array([base_lr * (0.1 ** (e // 30)) for e in epochs])

# 3. Exponential Decay (gamma=0.95)
lr_exp = np.array([base_lr * (0.95 ** e) for e in epochs])

# 4. Cosine Annealing
lr_cosine = np.array([
    min_lr + 0.5 * (base_lr - min_lr) * (1 + np.cos(np.pi * e / total_epochs))
    for e in epochs
])

# 5. Warmup (5 epochs) + Cosine Decay
warmup_epochs = 5
lr_warmup_cosine = []
for e in epochs:
    if e < warmup_epochs:
        lr_val = base_lr * (e + 1) / warmup_epochs
    else:
        progress = (e - warmup_epochs) / (total_epochs - warmup_epochs)
        lr_val = min_lr + 0.5 * (base_lr - min_lr) * (1 + np.cos(np.pi * progress))
    lr_warmup_cosine.append(lr_val)
lr_warmup_cosine = np.array(lr_warmup_cosine)

# 繪圖
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

schedules = [
    (lr_constant, 'Constant', '#95A5A6'),
    (lr_step, 'Step Decay (every 30, x0.1)', '#E74C3C'),
    (lr_exp, 'Exponential Decay (gamma=0.95)', '#3498DB'),
    (lr_cosine, 'Cosine Annealing', '#2ECC71'),
    (lr_warmup_cosine, 'Warmup + Cosine', '#9B59B6')
]

for idx, (lr_sched, name, color) in enumerate(schedules):
    ax = axes.flat[idx]
    ax.plot(epochs, lr_sched, color=color, linewidth=2.5)
    ax.fill_between(epochs, 0, lr_sched, color=color, alpha=0.15)
    ax.set_xlabel('Epoch', fontsize=10)
    ax.set_ylabel('Learning Rate', fontsize=10)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, base_lr * 1.1)

# 第六格：全部比較
ax_all = axes.flat[5]
for lr_sched, name, color in schedules:
    ax_all.plot(epochs, lr_sched, color=color, linewidth=2, label=name)
ax_all.set_xlabel('Epoch', fontsize=10)
ax_all.set_ylabel('Learning Rate', fontsize=10)
ax_all.set_title('All Schedules Compared', fontsize=11, fontweight='bold')
ax_all.legend(fontsize=7)
ax_all.grid(True, alpha=0.3)

plt.suptitle('Learning Rate Scheduling Strategies\n學習率排程策略', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('策略選擇指南 Strategy Guide:')
print('  Step Decay:     簡單經典，適合初學者')
print('  Exponential:    平滑衰減，但 gamma 敏感')
print('  Cosine:         後期衰減平緩，適合長時間訓練')
print('  Warmup+Cosine:  大模型/大 Batch 的首選')

---
## Part 4: 早停模擬 Early Stopping Simulation

早停是防止過擬合最直接的方法：
> **當驗證集上的效能不再改善時，停止訓練。**

關鍵參數：
- `patience`: 容忍幾個 epoch 無改善
- `min_delta`: 最小改善量

In [ ]:
# 模擬訓練過程 Simulate training process with overfitting
np.random.seed(42)
n_epochs_sim = 150
epochs_sim = np.arange(1, n_epochs_sim + 1)

# 模擬訓練損失（持續下降）
train_loss = 2.0 * np.exp(-0.03 * epochs_sim) + 0.05 * np.random.randn(n_epochs_sim) * np.exp(-0.02 * epochs_sim)
train_loss = np.maximum(train_loss, 0.05)

# 模擬驗證損失（先下降後上升 = 過擬合）
val_loss = 2.0 * np.exp(-0.025 * epochs_sim) + 0.003 * (epochs_sim - 50)**2 * (epochs_sim > 50).astype(float) / 100
val_loss += 0.08 * np.random.randn(n_epochs_sim) * np.exp(-0.01 * epochs_sim)
val_loss = np.maximum(val_loss, 0.1)

# 找到最佳和早停點
best_epoch = np.argmin(val_loss) + 1
best_val_loss = val_loss[best_epoch - 1]

# 模擬早停（patience=10）
patience = 10
min_delta = 0.005
best_so_far = float('inf')
counter = 0
early_stop_epoch = n_epochs_sim

for e in range(n_epochs_sim):
    if val_loss[e] < best_so_far - min_delta:
        best_so_far = val_loss[e]
        counter = 0
    else:
        counter += 1
    if counter >= patience:
        early_stop_epoch = e + 1
        break

# 繪圖
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左圖：無早停
ax = axes[0]
ax.plot(epochs_sim, train_loss, 'b-', linewidth=2, label='Training Loss', alpha=0.8)
ax.plot(epochs_sim, val_loss, 'r-', linewidth=2, label='Validation Loss', alpha=0.8)
ax.axvline(x=best_epoch, color='green', linestyle='--', linewidth=1.5,
           label=f'Best Epoch = {best_epoch}')
ax.fill_between(epochs_sim[best_epoch:], train_loss[best_epoch:], val_loss[best_epoch:],
                color='red', alpha=0.1, label='Generalization Gap')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.set_title('Without Early Stopping\n沒有早停 (訓練到底)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.annotate('Overfitting!\n過擬合！', xy=(130, val_loss[129]),
            xytext=(110, val_loss[129] + 0.5),
            fontsize=10, color='red', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='red'))

# 右圖：有早停
ax = axes[1]
ax.plot(epochs_sim[:early_stop_epoch], train_loss[:early_stop_epoch],
        'b-', linewidth=2, label='Training Loss', alpha=0.8)
ax.plot(epochs_sim[:early_stop_epoch], val_loss[:early_stop_epoch],
        'r-', linewidth=2, label='Validation Loss', alpha=0.8)
# 灰色虛線顯示如果繼續訓練
ax.plot(epochs_sim[early_stop_epoch-1:], train_loss[early_stop_epoch-1:],
        'b--', linewidth=1, alpha=0.3, label='(Would-be Training)')
ax.plot(epochs_sim[early_stop_epoch-1:], val_loss[early_stop_epoch-1:],
        'r--', linewidth=1, alpha=0.3, label='(Would-be Validation)')
ax.axvline(x=early_stop_epoch, color='orange', linestyle='-', linewidth=2.5,
           label=f'Early Stop at Epoch {early_stop_epoch} (patience={patience})')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.set_title('With Early Stopping\n使用早停', fontsize=12)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle('Early Stopping: Preventing Overfitting\n早停：防止過擬合', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f'Best validation loss at epoch {best_epoch}: {best_val_loss:.4f}')
print(f'Early stopping triggered at epoch {early_stop_epoch} (patience={patience})')
print(f'Without early stopping, final val loss: {val_loss[-1]:.4f} (overfitting!)')

---
## Part 5: 權重初始化 Weight Initialization

權重初始化對訓練穩定性至關重要：
- **Zero Init**: 所有神經元行為相同（對稱性問題）
- **Random Normal**: 可能導致梯度消失/爆炸
- **Xavier** (Glorot): $W \sim \mathcal{N}(0, \frac{2}{n_{in}+n_{out}})$ — 適合 tanh/sigmoid
- **He** (Kaiming): $W \sim \mathcal{N}(0, \frac{2}{n_{in}})$ — 適合 ReLU

In [ ]:
# 視覺化不同初始化方法的權重分布和前向傳播激活值
np.random.seed(42)
n_in = 256
n_out = 256
n_layers = 6

init_methods = {
    'Zero': lambda: np.zeros((n_in, n_out)),
    'Random Normal (std=1.0)': lambda: np.random.randn(n_in, n_out),
    'Random Normal (std=0.01)': lambda: np.random.randn(n_in, n_out) * 0.01,
    'Xavier / Glorot': lambda: np.random.randn(n_in, n_out) * np.sqrt(2.0 / (n_in + n_out)),
    'He / Kaiming': lambda: np.random.randn(n_in, n_out) * np.sqrt(2.0 / n_in),
}

fig, axes = plt.subplots(2, len(init_methods), figsize=(20, 8))

for col, (name, init_fn) in enumerate(init_methods.items()):
    # 上排：權重分布
    W = init_fn()
    axes[0, col].hist(W.flatten(), bins=50, density=True, alpha=0.7, color='steelblue')
    axes[0, col].set_title(f'{name}\nstd={W.std():.4f}', fontsize=9)
    axes[0, col].set_xlim(-3, 3)
    if col == 0:
        axes[0, col].set_ylabel('Density', fontsize=10)
    
    # 下排：經過多層前向傳播後的激活值分布
    x = np.random.randn(1, n_in)  # 一筆輸入
    activations_std = []
    for layer in range(n_layers):
        W_layer = init_fn()
        x = np.tanh(x @ W_layer)  # tanh activation
        activations_std.append(np.std(x))
    
    axes[1, col].bar(range(1, n_layers + 1), activations_std,
                      color='coral', alpha=0.8)
    axes[1, col].set_xlabel('Layer', fontsize=9)
    axes[1, col].set_title(f'Activation Std per Layer', fontsize=9)
    if col == 0:
        axes[1, col].set_ylabel('Std of Activations', fontsize=10)
    axes[1, col].set_ylim(0, 1.2)

plt.suptitle('Weight Initialization Methods: Distribution & Forward Pass Activations\n'
             '權重初始化方法：分布與前向傳播激活值', fontsize=13, y=1.03)
plt.tight_layout()
plt.show()

print('觀察重點 Key Observations:')
print('  Zero:           所有激活為 0，無法學習（對稱性問題）')
print('  Large Random:   激活飽和，梯度消失')
print('  Small Random:   激活值逐層衰減至 0')
print('  Xavier/He:      激活值在各層保持穩定 — 最理想！')

---
## Part 6: 資料增強 Data Augmentation Concepts

資料增強透過對原始資料施加隨機變換，產生更多訓練樣本，降低過擬合。

常見的影像增強方法（使用 numpy/scipy 實作）：
- 旋轉 (Rotation)
- 翻轉 (Flip)
- 高斯噪音 (Gaussian Noise)
- 平移 (Shift)
- 模糊 (Blur)

In [ ]:
# 使用 sklearn 的 digits 資料集進行資料增強示範
from sklearn.datasets import load_digits
digits = load_digits()

# 選擇幾個範例數字
sample_indices = [0, 10, 20]  # digits 0, 1, 2

def augment_image(img, method):
    """Apply augmentation to an 8x8 digit image."""
    img_2d = img.reshape(8, 8)
    
    if method == 'original':
        return img_2d
    elif method == 'rotate_15':
        return rotate(img_2d, 15, reshape=False, mode='constant', cval=0)
    elif method == 'rotate_-15':
        return rotate(img_2d, -15, reshape=False, mode='constant', cval=0)
    elif method == 'flip_h':
        return np.fliplr(img_2d)
    elif method == 'flip_v':
        return np.flipud(img_2d)
    elif method == 'noise':
        noise = np.random.randn(*img_2d.shape) * 2
        return np.clip(img_2d + noise, 0, 16)
    elif method == 'shift_right':
        return shift(img_2d, [0, 1], mode='constant', cval=0)
    elif method == 'shift_down':
        return shift(img_2d, [1, 0], mode='constant', cval=0)
    elif method == 'blur':
        return gaussian_filter(img_2d, sigma=0.8)
    
    return img_2d

augment_methods = ['original', 'rotate_15', 'rotate_-15', 'flip_h',
                   'noise', 'shift_right', 'shift_down', 'blur']
method_labels = ['Original', 'Rotate +15', 'Rotate -15', 'Flip H',
                 'Gaussian Noise', 'Shift Right', 'Shift Down', 'Blur']

fig, axes = plt.subplots(len(sample_indices), len(augment_methods), figsize=(20, 8))

for row, idx in enumerate(sample_indices):
    img = digits.data[idx]
    for col, (method, label) in enumerate(zip(augment_methods, method_labels)):
        np.random.seed(42)  # 固定噪音
        aug_img = augment_image(img, method)
        axes[row, col].imshow(aug_img, cmap='gray_r', interpolation='nearest')
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(label, fontsize=9, fontweight='bold')
    axes[row, 0].set_ylabel(f'Digit {digits.target[idx]}', fontsize=11, rotation=0,
                             labelpad=30)

plt.suptitle('Data Augmentation on Digit Images\n數字影像的資料增強示範',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('資料增強注意事項 Data Augmentation Tips:')
print('  1. 只對訓練集做增強，測試集不做！')
print('  2. 增強後的資料應仍屬於原始類別（語意不變）')
print('  3. 過度增強可能反而有害')
print('  4. 注意領域知識：醫學影像不宜隨意翻轉')

---
## Part 7: Batch Size 效果 Batch Size Effect

Batch Size 影響：
- **大 Batch**：梯度估計更準確，但每步計算更慢，泛化可能較差
- **小 Batch**：梯度有噪音（隱式正則化），但收斂路徑更不穩定

我們模擬不同 Batch Size 下的梯度噪音和收斂行為。

In [ ]:
# 模擬不同 batch size 對收斂的影響
np.random.seed(42)

# 生成分類資料
X_bs, y_bs = make_moons(n_samples=800, noise=0.25, random_state=42)
X_train_bs, X_test_bs, y_train_bs, y_test_bs = train_test_split(
    X_bs, y_bs, test_size=0.3, random_state=42
)
scaler_bs = StandardScaler()
X_train_bs = scaler_bs.fit_transform(X_train_bs)
X_test_bs = scaler_bs.transform(X_test_bs)

batch_sizes = [8, 32, 128, 560]  # 560 = nearly full batch
batch_labels = ['BS=8 (Mini)', 'BS=32', 'BS=128', 'BS=560 (Full)']
colors_bs = ['#E74C3C', '#3498DB', '#2ECC71', '#9B59B6']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for bs, label, color in zip(batch_sizes, batch_labels, colors_bs):
    mlp_bs = MLPClassifier(
        hidden_layer_sizes=(32, 16), max_iter=200,
        learning_rate_init=0.01, learning_rate='constant',
        solver='sgd', random_state=42, batch_size=min(bs, len(X_train_bs))
    )
    mlp_bs.fit(X_train_bs, y_train_bs)
    
    axes[0].plot(mlp_bs.loss_curve_, color=color, linewidth=1.5,
                 label=label, alpha=0.8)
    test_acc = mlp_bs.score(X_test_bs, y_test_bs)
    axes[1].bar(label, test_acc, color=color, alpha=0.8)

axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Training Loss', fontsize=11)
axes[0].set_title('Training Loss with Different Batch Sizes\n不同 Batch Size 的訓練損失', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].set_ylabel('Test Accuracy', fontsize=11)
axes[1].set_title('Test Accuracy vs Batch Size\nBatch Size vs 測試準確率', fontsize=12)
axes[1].set_ylim(0.75, 1.0)
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Batch Size Effect on Training\nBatch Size 對訓練的影響', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('觀察重點 Key Observations:')
print('  小 Batch: 損失曲線有較多噪音（隱式正則化效果）')
print('  大 Batch: 損失曲線較平滑，但可能泛化能力稍差')
print('  實務建議: BS=32~256 通常是好的起點')

---
## Part 8: 梯度裁剪 Gradient Clipping Visualization

梯度爆炸時，梯度值變得非常大，導致參數更新過猛。

**按範數裁剪 Clip by Norm**（推薦）：
$$\hat{g} = \begin{cases} g & \text{if } \|g\| \leq \text{max\_norm} \\ \frac{\text{max\_norm}}{\|g\|} g & \text{if } \|g\| > \text{max\_norm} \end{cases}$$

保持梯度方向不變，只縮放大小。

In [ ]:
# 梯度裁剪視覺化 Gradient clipping visualization
np.random.seed(42)

# 模擬一個訓練過程中的梯度範數序列
n_steps_gc = 100
# 模擬：正常梯度 + 偶爾的梯度爆炸
gradient_norms = np.abs(np.random.randn(n_steps_gc) * 1.5)
# 插入幾個梯度爆炸
spike_indices = [15, 35, 52, 70, 88]
for idx in spike_indices:
    gradient_norms[idx] = np.random.uniform(8, 20)

max_norm = 5.0

# 裁剪後的梯度
clipped_norms = np.minimum(gradient_norms, max_norm)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 左圖：未裁剪的梯度範數
axes[0].bar(range(n_steps_gc), gradient_norms, color='royalblue', alpha=0.7)
for idx in spike_indices:
    axes[0].bar(idx, gradient_norms[idx], color='red', alpha=0.9)
axes[0].axhline(y=max_norm, color='orange', linestyle='--', linewidth=2,
                label=f'max_norm = {max_norm}')
axes[0].set_xlabel('Training Step', fontsize=10)
axes[0].set_ylabel('Gradient Norm', fontsize=10)
axes[0].set_title('Before Clipping\n裁剪前', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3, axis='y')

# 中圖：裁剪後的梯度範數
axes[1].bar(range(n_steps_gc), clipped_norms, color='forestgreen', alpha=0.7)
for idx in spike_indices:
    axes[1].bar(idx, clipped_norms[idx], color='orange', alpha=0.9)
axes[1].axhline(y=max_norm, color='orange', linestyle='--', linewidth=2,
                label=f'max_norm = {max_norm}')
axes[1].set_xlabel('Training Step', fontsize=10)
axes[1].set_ylabel('Gradient Norm', fontsize=10)
axes[1].set_title('After Clipping\n裁剪後', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_ylim(0, max(gradient_norms) * 1.1)

# 右圖：2D 梯度向量裁剪示意
ax = axes[2]
n_arrows = 20
np.random.seed(123)
gradients_2d = np.random.randn(n_arrows, 2) * 3
# 加幾個大梯度
gradients_2d[0] = [8, 6]
gradients_2d[5] = [-7, 9]
gradients_2d[10] = [10, -5]

clip_radius = 5.0
circle = plt.Circle((0, 0), clip_radius, fill=False, color='orange',
                     linewidth=2, linestyle='--', label=f'Clip radius={clip_radius}')
ax.add_patch(circle)

for g in gradients_2d:
    norm_g = np.linalg.norm(g)
    if norm_g > clip_radius:
        g_clipped = g * clip_radius / norm_g
        ax.arrow(0, 0, g[0], g[1], head_width=0.2, head_length=0.15,
                 fc='red', ec='red', alpha=0.3, linewidth=0.5)
        ax.arrow(0, 0, g_clipped[0], g_clipped[1], head_width=0.2, head_length=0.15,
                 fc='green', ec='green', alpha=0.8, linewidth=1.5)
    else:
        ax.arrow(0, 0, g[0], g[1], head_width=0.2, head_length=0.15,
                 fc='steelblue', ec='steelblue', alpha=0.7, linewidth=1)

ax.set_xlim(-12, 12)
ax.set_ylim(-12, 12)
ax.set_aspect('equal')
ax.set_xlabel('Gradient dim 1', fontsize=10)
ax.set_ylabel('Gradient dim 2', fontsize=10)
ax.set_title('Clip by Norm (2D view)\n按範數裁剪', fontsize=12)
ax.legend(fontsize=9, loc='upper left')
ax.grid(True, alpha=0.3)

# 自訂圖例
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='steelblue', lw=2, label='Normal gradient'),
    Line2D([0], [0], color='red', lw=2, alpha=0.3, label='Before clipping'),
    Line2D([0], [0], color='green', lw=2, label='After clipping'),
]
ax.legend(handles=legend_elements + [circle], fontsize=8, loc='upper left')

plt.suptitle('Gradient Clipping Visualization\n梯度裁剪視覺化', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('梯度裁剪要點 Gradient Clipping Tips:')
print('  - 按範數裁剪保持梯度方向，只縮放大小')
print('  - 常用 max_norm: 0.5~5.0')
print('  - RNN/LSTM 訓練幾乎必備')
print('  - Transformer 通常使用 max_norm=1.0')

---
## Part 9: 訓練技巧綜合比較 Training Tricks Summary

把所有技巧整合到一個模擬訓練迴圈中，比較使用前後的效果差異。

In [ ]:
# 綜合比較：使用 sklearn MLPClassifier 在 digits 資料集上
np.random.seed(42)
digits = load_digits()
X_d, y_d = digits.data, digits.target

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_d, y_d, test_size=0.3, random_state=42, stratify=y_d
)

scaler_d = StandardScaler()
X_train_d = scaler_d.fit_transform(X_train_d)
X_test_d = scaler_d.transform(X_test_d)

# 模擬不同訓練配置
configs = {
    'Baseline (SGD, lr=0.01)': {
        'solver': 'sgd', 'learning_rate_init': 0.01,
        'learning_rate': 'constant', 'early_stopping': False,
        'max_iter': 300, 'batch_size': 200
    },
    '+ Momentum (Adam)': {
        'solver': 'adam', 'learning_rate_init': 0.001,
        'learning_rate': 'constant', 'early_stopping': False,
        'max_iter': 300, 'batch_size': 200
    },
    '+ LR Schedule (adaptive)': {
        'solver': 'adam', 'learning_rate_init': 0.001,
        'learning_rate': 'adaptive', 'early_stopping': False,
        'max_iter': 300, 'batch_size': 200
    },
    '+ Early Stopping': {
        'solver': 'adam', 'learning_rate_init': 0.001,
        'learning_rate': 'adaptive', 'early_stopping': True,
        'max_iter': 300, 'batch_size': 200, 'n_iter_no_change': 15,
        'validation_fraction': 0.15
    },
    '+ Smaller Batch (32)': {
        'solver': 'adam', 'learning_rate_init': 0.001,
        'learning_rate': 'adaptive', 'early_stopping': True,
        'max_iter': 300, 'batch_size': 32, 'n_iter_no_change': 15,
        'validation_fraction': 0.15
    },
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors_cfg = plt.cm.tab10(np.linspace(0, 0.5, len(configs)))

test_results = []

for (name, cfg), color in zip(configs.items(), colors_cfg):
    mlp = MLPClassifier(
        hidden_layer_sizes=(64, 32),
        random_state=42,
        **cfg
    )
    mlp.fit(X_train_d, y_train_d)
    
    train_acc = mlp.score(X_train_d, y_train_d)
    test_acc = mlp.score(X_test_d, y_test_d)
    n_iters = len(mlp.loss_curve_)
    test_results.append((name, train_acc, test_acc, n_iters))
    
    axes[0].plot(mlp.loss_curve_, color=color, linewidth=1.5,
                 label=f'{name} ({n_iters} iters)', alpha=0.8)

axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Loss', fontsize=11)
axes[0].set_title('Training Loss Curves\n各配置的訓練損失曲線', fontsize=12)
axes[0].legend(fontsize=7, loc='upper right')
axes[0].grid(True, alpha=0.3)

# 右圖：Test Accuracy 比較
names_short = [n.replace(' ', '\n') for n, _, _, _ in test_results]
test_accs_cfg = [ta for _, _, ta, _ in test_results]
bars = axes[1].barh(range(len(test_results)), test_accs_cfg, color=colors_cfg, alpha=0.8)
axes[1].set_yticks(range(len(test_results)))
axes[1].set_yticklabels([n for n, _, _, _ in test_results], fontsize=8)
axes[1].set_xlabel('Test Accuracy', fontsize=11)
axes[1].set_title('Test Accuracy Comparison\n測試準確率比較', fontsize=12)
axes[1].set_xlim(0.9, 1.0)
for bar, acc in zip(bars, test_accs_cfg):
    axes[1].text(acc + 0.001, bar.get_y() + bar.get_height()/2,
                 f'{acc:.4f}', va='center', fontsize=9)
axes[1].grid(True, alpha=0.3, axis='x')

plt.suptitle('Training Tricks Comparison on Digits Dataset\n'
             '訓練技巧在手寫數字資料集上的綜合比較', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('\n=== Results Summary ===')
print(f'{"Config":<35} {"Train Acc":<12} {"Test Acc":<12} {"Epochs":<8}')
print('=' * 67)
for name, train_a, test_a, n_it in test_results:
    print(f'{name:<35} {train_a:.4f}       {test_a:.4f}       {n_it}')

---
## Exercises 練習題

完成以下練習來鞏固本週所學。

### Exercise 1: 實作 Cosine Annealing with Warm Restarts

實作 SGDR (Stochastic Gradient Descent with Warm Restarts)：
學習率按餘弦衰減到最小值後，**重新回彈**到較大值。

參數：
- `T_0`: 第一個週期的長度
- `T_mult`: 每個週期長度的倍增因子
- 總共繪製 100 個 epoch 的學習率曲線

In [ ]:
# Exercise 1: 請在此撰寫你的程式碼
# TODO: Implement Cosine Annealing with Warm Restarts

# def cosine_warm_restarts(epoch, T_0=10, T_mult=2, base_lr=0.1, min_lr=1e-5):
#     """Compute learning rate with warm restarts."""
#     # Hint: Track which cycle we're in, and what the current T is
#     # Cycle 0: T=T_0, Cycle 1: T=T_0*T_mult, Cycle 2: T=T_0*T_mult^2, ...
#     pass

# epochs = np.arange(100)
# lrs = [cosine_warm_restarts(e) for e in epochs]
# plt.plot(epochs, lrs)
# plt.xlabel('Epoch')
# plt.ylabel('Learning Rate')
# plt.title('Cosine Annealing with Warm Restarts')
# plt.show()

### Exercise 2: 找最佳早停 Epoch

使用 `make_classification` 生成 2000 個樣本的資料集，
訓練一個 `MLPClassifier`，記錄每個 epoch 的 validation loss，
手動實作早停邏輯找到最佳停止點。

嘗試不同的 patience 值 (5, 10, 20)，觀察效果差異。

In [ ]:
# Exercise 2: 請在此撰寫你的程式碼
# TODO: Find optimal early stopping epoch with different patience values

# Hint: Use MLPClassifier with early_stopping=True and different n_iter_no_change values
# X_es, y_es = make_classification(n_samples=2000, n_features=20, random_state=42)
# ...
# for patience in [5, 10, 20]:
#     mlp = MLPClassifier(..., early_stopping=True, n_iter_no_change=patience)
#     ...

### Exercise 3: 比較 Adam vs SGD+Momentum

在 `load_digits` 資料集上比較：
1. Adam (lr=0.001)
2. SGD + Momentum (lr=0.01, momentum=0.9)

繪製兩者的訓練曲線和測試準確率，並記錄收斂所需的 epoch 數。

哪個優化器收斂更快？哪個最終準確率更高？

In [ ]:
# Exercise 3: 請在此撰寫你的程式碼
# TODO: Compare Adam vs SGD+Momentum on digits dataset

# adam_mlp = MLPClassifier(solver='adam', learning_rate_init=0.001, ...)
# sgd_mlp = MLPClassifier(solver='sgd', learning_rate_init=0.01, momentum=0.9, ...)
# ...
# Plot training curves and compare

---
## 本週實作總結 Lab Summary

在這次實作中，我們完成了：

1. **優化器比較**：SGD、Momentum、RMSProp、Adam 在 Rosenbrock 函數上的路徑
2. **學習率效果**：過大/過小的學習率對訓練的影響
3. **學習率排程**：Constant、Step、Exponential、Cosine、Warmup+Cosine
4. **早停機制**：防止過擬合的最直接方法
5. **權重初始化**：Xavier/He 保持激活值穩定
6. **資料增強**：旋轉、翻轉、噪音、平移、模糊
7. **Batch Size**：小 Batch 有隱式正則化效果
8. **梯度裁剪**：防止梯度爆炸
9. **綜合比較**：疊加技巧逐步提升模型表現

### 關鍵收穫 Key Takeaways
- 學習率是最重要的超參數，**動態排程**優於固定值
- 早停 + 最佳模型儲存 = 防過擬合的第一道防線
- 權重初始化影響訓練穩定性，Xavier/He 是標準選擇
- 資料增強是「免費的正則化」
- Adam 通常是好的預設優化器，SGD+Momentum 在精調時可能更好

### 下週預告 Next Week Preview
**第 15 週：模型評估與公平性 (Model Evaluation & Fairness)**
- 進階評估指標（多類別混淆矩陣、校準曲線）
- 公平性指標（Demographic Parity、Equalized Odds）
- 偏誤檢測與緩解
- 模型穩健性測試